In [ ]:
!nvidia-smi | head -12

# GPU benchmark: TokenVector.Inference vs PyTorch trên Colab GPU

So sánh train GPT-2 124M (random init, không cần fixture) forward+backward trên:
- **.tkv** — `demo/bseq.dll` (IL portable) + `tkv_simd_native` (C, AVX2) + dispatch cuBLAS nếu `libcublas.so.12` load được
- **PyTorch** — CUDA

Cùng seq1024, mode 0 = fwd+bwd (không optimizer) để apple-to-apple.

> **Trạng thái 2026-11-12 (đo trên T4):** `TkvCUDA_Available() = 1` (cuBLAS nạp được)
> nhưng GPU **thực tế không được dùng** — hook cuBLAS chỉ nằm trong kernel tile
> `SgemmF32(a,b,o,T)` với `T < 512`, và mỗi lần gọi lại `cudaMalloc` + copy.
> Kết quả `.tkv` dưới đây là **CPU** (2 vCPU của Colab), không phải GPU.
>
> Trên Linux, kernel ASM AVX2 (Windows x64 ABI) không dùng được — engine tự rơi về
> bản vô hướng cho các op phần tử, còn GEMM/attention vẫn chạy trong C `.so`.
> Phép so sánh đúng là **.tkv CPU** vs **PyTorch CPU** (cùng VM); số PyTorch
> CUDA chỉ để tham khảo.

In [ ]:
# [1/4] Install .NET 10 SDK (full SDK, includes runtime) + required libs
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
!curl -fsSL https://dot.net/v1/dotnet-install.sh -o /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --info | head -15


In [ ]:
# [2/4] Clone repos + CUDA runtime libs (cublas/cudart) + LD_LIBRARY_PATH
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git
!pip install -q nvidia-cublas-cu12 nvidia-cuda-runtime-cu12 2>&1 | tail -2
import os, glob
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
if cublas_dir:
    os.environ['LD_LIBRARY_PATH'] = cublas_dir + ':' + (cudart_dir or '') + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# [3/4] Build native .so (C + AVX2 + CUDA dispatch) và kiểm tra CUDA khả dụng
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/demo/tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/tkv_simd_native.dll
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll
!cp TokenVector/Release/tools/simd_native/tkv_step124.dll TokenVector.Inference/demo/tkv_step124.dll
print('Built .so')
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1 = cuBLAS loaded, 0 = CPU fallback)')


In [ ]:
# [4/4] Bench .tkv vs PyTorch (seq1024, fwd+bwd) — CPU và CUDA
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt
# sampler có timeout (KHÔNG dùng pkill -f nvidia-smi: nó giết luôn shell của cell)
!nohup timeout 400 nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader --loop=2 > /content/gpu.log 2>&1 & echo sampler_started
!cd TokenVector.Inference && /opt/dotnet/dotnet demo/bseq.dll 2>&1 | tail -3
!echo '--- GPU trong luc .tkv chay (moi dong = 1 mau 2s) ---'; sort -u /content/gpu.log | head -6
print('--- PyTorch CPU (cung VM, apples-to-apples voi .tkv CPU) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu python3 benchmark/bench_torch.py d 1024 2>&1 | tail -3
print('--- PyTorch CUDA (T4) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024 2>&1 | tail -3
!echo '--- GPU trong luc torch CUDA chay ---'; tail -12 /content/gpu.log
!nvidia-smi --query-gpu=memory.used --format=csv,noheader
